# Target Encoding

Target Encoding is a categorical encoding technique that replaces each category with a numerical value calculated from the target variable.

For a classification problem, a category is often replaced by the average target value for that category.

For example:

If:

Lahore → 0.80
Karachi → 0.50
Islamabad → 0.30

then these values represent the average target rate for each city.

Target Encoding is particularly useful for categorical features with many unique categories.

In [1]:
import pandas as pd

df = pd.DataFrame({
    "City": [
        "Lahore", "Lahore", "Lahore",
        "Karachi", "Karachi", "Karachi",
        "Islamabad", "Islamabad", "Islamabad"
    ],
    "Purchased": [
        1, 1, 0,
        1, 0, 0,
        0, 0, 1
    ]
})

df

,City,Purchased
0,Lahore,1
1,Lahore,1
2,Lahore,0
3,Karachi,1
4,Karachi,0
5,Karachi,0
6,Islamabad,0
7,Islamabad,0
8,Islamabad,1


#### Understanding the Target Variable

Our target variable is:

`Purchased`

where:

- `1` = Customer purchased
- `0` = Customer did not purchase

We want to encode the `City` feature based on the average purchase rate for each city.

In [2]:
target_mean = df.groupby("City")["Purchased"].mean()

target_mean

City
Islamabad    0.333333
Karachi      0.333333
Lahore       0.666667
Name: Purchased, dtype: float64

#### Manual Target Encoding

We can replace each category with its corresponding target mean.

For example:

Lahore:

(1 + 1 + 0) / 3 = 0.67

Karachi:

(1 + 0 + 0) / 3 = 0.33

Islamabad:

(0 + 0 + 1) / 3 = 0.33

In [3]:
df["City_Target_Encoded"] = df["City"].map(target_mean)

df

,City,Purchased,City_Target_Encoded
0,Lahore,1,0.666667
1,Lahore,1,0.666667
2,Lahore,0,0.666667
3,Karachi,1,0.333333
4,Karachi,0,0.333333
5,Karachi,0,0.333333
6,Islamabad,0,0.333333
7,Islamabad,0,0.333333
8,Islamabad,1,0.333333


In [4]:
target_mapping = (
    df.groupby("City")["Purchased"]
    .mean()
    .to_dict()
)

target_mapping

{'Islamabad': 0.3333333333333333,
 'Karachi': 0.3333333333333333,
 'Lahore': 0.6666666666666666}

#### Why Use Target Encoding?

Target Encoding can be useful when a categorical feature contains many unique categories.

For example:

- City
- ZIP Code
- Product ID
- Customer ID
- Store ID

One-Hot Encoding can create hundreds or thousands of columns for high-cardinality features.

Target Encoding represents each category with a small number of numerical features instead.

#### Using Category Encoders

The `category_encoders` library provides a convenient implementation of Target Encoding.

It can be installed using:

`pip install category_encoders`

In [7]:
import category_encoders as ce

In [8]:
encoder = ce.TargetEncoder(
    cols=["City"]
)

df_encoded = encoder.fit_transform(
    df[["City"]],
    df["Purchased"]
)

df_encoded

,City
0,0.478770
1,0.478770
2,0.478770
3,0.427282
4,0.427282
5,0.427282
6,0.427282
7,0.427282
8,0.427282


In [9]:
result = pd.concat(
    [
        df[["City", "Purchased"]],
        df_encoded.rename(columns={"City": "City_Target_Encoded"})
    ],
    axis=1
)

result

,City,Purchased,City_Target_Encoded
0,Lahore,1,0.478770
1,Lahore,1,0.478770
2,Lahore,0,0.478770
3,Karachi,1,0.427282
4,Karachi,0,0.427282
5,Karachi,0,0.427282
6,Islamabad,0,0.427282
7,Islamabad,0,0.427282
8,Islamabad,1,0.427282


#### Target Encoding for Regression

Target Encoding can also be used for regression problems.

Instead of calculating the average probability of a class, we calculate the average numerical target for each category.

For example, suppose we want to predict house prices based on city.

Each city can be replaced by its average house price.

In [10]:
house_df = pd.DataFrame({
    "City": [
        "Lahore", "Lahore",
        "Karachi", "Karachi",
        "Islamabad", "Islamabad"
    ],
    "Price": [
        12000000, 15000000,
        9000000, 11000000,
        18000000, 20000000
    ]
})

house_df

,City,Price
0,Lahore,12000000
1,Lahore,15000000
2,Karachi,9000000
3,Karachi,11000000
4,Islamabad,18000000
5,Islamabad,20000000


In [11]:
city_price_mean = (
    house_df.groupby("City")["Price"]
    .mean()
)

city_price_mean

City
Islamabad    19000000.0
Karachi      10000000.0
Lahore       13500000.0
Name: Price, dtype: float64

In [12]:
house_df["City_Encoded"] = (
    house_df["City"].map(city_price_mean)
)

house_df

,City,Price,City_Encoded
0,Lahore,12000000,13500000.0
1,Lahore,15000000,13500000.0
2,Karachi,9000000,10000000.0
3,Karachi,11000000,10000000.0
4,Islamabad,18000000,19000000.0
5,Islamabad,20000000,19000000.0
